In [1]:
import pandas as pd
import numpy as np

# =========================
# 1. データ読み込み
# =========================

customers = pd.read_csv("data/customers.csv")
products = pd.read_csv("data/products.csv")
stores = pd.read_csv("data/stores.csv")
orders = pd.read_csv("data/orders.csv")

print("customers:", customers.shape)
print("products:", products.shape)
print("stores:", stores.shape)
print("orders:", orders.shape)

FileNotFoundError: [Errno 2] No such file or directory: 'data/customers.csv'

In [ ]:
# =========================
# 2. データ構造の確認
# =========================

print("=== customers ===")
display(customers.head())

print("=== products ===")
display(products.head())

print("=== stores ===")
display(stores.head())

print("=== orders ===")
display(orders.head())

=== customers ===


NameError: name 'customers' is not defined

In [ ]:
# =========================
# 3. 欠損値チェック
# =========================

print("=== Missing Values ===")

print("customers")
display(customers.isnull().sum())

print("products")
display(products.isnull().sum())

print("stores")
display(stores.isnull().sum())

print("orders")
display(orders.isnull().sum())

=== Missing Values ===
customers


NameError: name 'customers' is not defined

In [ ]:
# =========================
# 4. ID重複チェック
# =========================

print("customers duplicate customer_id:",
      customers["customer_id"].duplicated().sum())

print("products duplicate product_id:",
      products["product_id"].duplicated().sum())

print("stores duplicate store_id:",
      stores["store_id"].duplicated().sum())

print("orders duplicate order_id:",
      orders["order_id"].duplicated().sum())

NameError: name 'customers' is not defined

In [ ]:
# =========================
# 5. データ型の変換
# =========================

orders["order_date"] = pd.to_datetime(
    orders["order_date"],
    errors="coerce"
)

orders["quantity"] = pd.to_numeric(
    orders["quantity"],
    errors="coerce"
)

orders["unit_price"] = pd.to_numeric(
    orders["unit_price"],
    errors="coerce"
)

customers["birth_year"] = pd.to_numeric(
    customers["birth_year"],
    errors="coerce"
)

print("=== Data Types ===")
display(orders.dtypes)

NameError: name 'orders' is not defined

In [ ]:
# =========================
# 6. 異常値チェック
# =========================

print("=== Invalid Values Check ===")

print(
    "Invalid order_date:",
    orders["order_date"].isna().sum()
)

print(
    "Invalid quantity:",
    orders["quantity"].isna().sum()
)

print(
    "Invalid unit_price:",
    orders["unit_price"].isna().sum()
)

print(
    "Quantity <= 0:",
    (orders["quantity"] <= 0).sum()
)

print(
    "Unit price <= 0:",
    (orders["unit_price"] <= 0).sum()
)

In [ ]:
# =========================
# 7. Referential Integrity
# =========================

invalid_customers = orders[
    ~orders["customer_id"].isin(customers["customer_id"])
]

invalid_products = orders[
    ~orders["product_id"].isin(products["product_id"])
]

invalid_stores = orders[
    ~orders["store_id"].isin(stores["store_id"])
]

print(
    "Orders with invalid customer_id:",
    len(invalid_customers)
)

print(
    "Orders with invalid product_id:",
    len(invalid_products)
)

print(
    "Orders with invalid store_id:",
    len(invalid_stores)
)

In [ ]:
# =========================
# 8. Sales Amount
# =========================

orders["sales_amount"] = (
    orders["quantity"] * orders["unit_price"]
)

display(
    orders[
        [
            "order_id",
            "quantity",
            "unit_price",
            "sales_amount"
        ]
    ].head(10)
)

In [ ]:
# =========================
# 9. Date Features
# =========================

orders["year"] = orders["order_date"].dt.year
orders["month"] = orders["order_date"].dt.month
orders["year_month"] = (
    orders["order_date"]
    .dt.to_period("M")
    .astype(str)
)

display(
    orders[
        [
            "order_date",
            "year",
            "month",
            "year_month"
        ]
    ].head()
)

In [ ]:
# =========================
# 10. Join Customers
# =========================

analysis_df = orders.merge(
    customers,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

print("analysis_df shape:", analysis_df.shape)

display(analysis_df.head())

In [ ]:
# =========================
# 11. Join Products
# =========================

analysis_df = analysis_df.merge(
    products,
    on="product_id",
    how="left",
    validate="many_to_one",
    suffixes=("", "_product")
)

print("analysis_df shape:", analysis_df.shape)

display(analysis_df.head())

In [ ]:
# =========================
# 12. Join Stores
# =========================

analysis_df = analysis_df.merge(
    stores,
    on="store_id",
    how="left",
    validate="many_to_one",
    suffixes=("", "_store")
)

print("analysis_df shape:", analysis_df.shape)

display(analysis_df.head())

In [ ]:
# =========================
# 13. Post-Join Validation
# =========================

print("=== Missing Values After Join ===")

display(
    analysis_df.isnull().sum()
    .sort_values(ascending=False)
)

In [ ]:
# =========================
# 14. Analysis Dataset
# =========================

print("Rows:", len(analysis_df))
print("Columns:", len(analysis_df.columns))

display(
    analysis_df[
        [
            "order_id",
            "customer_id",
            "product_id",
            "store_id",
            "order_date",
            "quantity",
            "unit_price",
            "sales_amount",
            "sales_channel"
        ]
    ].head(10)
)

In [ ]:
# =========================
# 15. Annual KPI
# =========================

annual_kpi = (
    analysis_df
    .groupby("year")
    .agg(
        total_sales=("sales_amount", "sum"),
        total_orders=("order_id", "nunique"),
        unique_customers=("customer_id", "nunique"),
        total_quantity=("quantity", "sum")
    )
    .reset_index()
)

# 平均注文金額
annual_kpi["average_order_value"] = (
    annual_kpi["total_sales"]
    / annual_kpi["total_orders"]
)

# 1人あたり年間購入金額
annual_kpi["annual_spend_per_customer"] = (
    annual_kpi["total_sales"]
    / annual_kpi["unique_customers"]
)

# 1人あたり購入点数
annual_kpi["items_per_customer"] = (
    annual_kpi["total_quantity"]
    / annual_kpi["unique_customers"]
)

display(annual_kpi)

In [ ]:
# =========================
# 16. KPI Formatting
# =========================

annual_kpi_display = annual_kpi.copy()

annual_kpi_display["total_sales"] = (
    annual_kpi_display["total_sales"].round(2)
)

annual_kpi_display["average_order_value"] = (
    annual_kpi_display["average_order_value"].round(2)
)

annual_kpi_display["annual_spend_per_customer"] = (
    annual_kpi_display["annual_spend_per_customer"].round(2)
)

annual_kpi_display["items_per_customer"] = (
    annual_kpi_display["items_per_customer"].round(2)
)

display(annual_kpi_display)

In [ ]:
# =========================
# 17. KPI Definition Check
# =========================

print("=== KPI Definitions ===")
print("Total Sales = quantity × unit_price の合計")
print("Orders = ユニークな order_id")
print("Customers = ユニークな customer_id")
print("AOV = Total Sales ÷ Orders")
print("Annual Spend per Customer = Total Sales ÷ Customers")
print("Items per Customer = Total Quantity ÷ Customers")

In [ ]:
# =========================
# 18. YoY Analysis
# =========================

yoy_kpi = annual_kpi.copy()

kpi_columns = [
    "total_sales",
    "total_orders",
    "unique_customers",
    "total_quantity",
    "average_order_value",
    "annual_spend_per_customer",
    "items_per_customer"
]

for col in kpi_columns:
    yoy_col = f"{col}_yoy"

    yoy_kpi[yoy_col] = (
        yoy_kpi[col].pct_change() * 100
    )

yoy_kpi = yoy_kpi.round(2)

display(yoy_kpi)

NameError: name 'annual_kpi' is not defined

In [ ]:
# =========================
# 19. YoY Summary
# =========================

yoy_summary = yoy_kpi[
    [
        "year",
        "total_sales_yoy",
        "total_orders_yoy",
        "unique_customers_yoy",
        "total_quantity_yoy",
        "average_order_value_yoy",
        "annual_spend_per_customer_yoy",
        "items_per_customer_yoy"
    ]
].copy()

display(yoy_summary)

In [ ]:
# =========================
# 20. 2026 YoY Result
# =========================

yoy_2026 = yoy_kpi[
    yoy_kpi["year"] == 2026
].copy()

display(
    yoy_2026[
        [
            "year",
            "total_sales_yoy",
            "total_orders_yoy",
            "unique_customers_yoy",
            "total_quantity_yoy",
            "average_order_value_yoy",
            "annual_spend_per_customer_yoy",
            "items_per_customer_yoy"
        ]
    ]
)

In [ ]:
# =========================
# 21. Revenue Driver Decomposition
# =========================

decomposition = annual_kpi[
    [
        "year",
        "total_sales",
        "unique_customers",
        "annual_spend_per_customer",
        "total_quantity",
        "items_per_customer"
    ]
].copy()

decomposition["price_per_item"] = (
    decomposition["total_sales"]
    / decomposition["total_quantity"]
)

display(
    decomposition.style.format({
        "total_sales": "${:,.2f}",
        "annual_spend_per_customer": "${:,.2f}",
        "price_per_item": "${:,.2f}"
    })
)

In [ ]:
# =========================
# 22. Driver YoY
# =========================

driver_yoy = pd.DataFrame({
    "metric": [
        "Total Sales",
        "Unique Customers",
        "Annual Spend per Customer",
        "Items per Customer",
        "Price per Item"
    ],
    "2025": [
        decomposition.loc[
            decomposition["year"] == 2025,
            "total_sales"
        ].iloc[0],

        decomposition.loc[
            decomposition["year"] == 2025,
            "unique_customers"
        ].iloc[0],

        decomposition.loc[
            decomposition["year"] == 2025,
            "annual_spend_per_customer"
        ].iloc[0],

        decomposition.loc[
            decomposition["year"] == 2025,
            "items_per_customer"
        ].iloc[0],

        decomposition.loc[
            decomposition["year"] == 2025,
            "price_per_item"
        ].iloc[0]
    ],
    "2026": [
        decomposition.loc[
            decomposition["year"] == 2026,
            "total_sales"
        ].iloc[0],

        decomposition.loc[
            decomposition["year"] == 2026,
            "unique_customers"
        ].iloc[0],

        decomposition.loc[
            decomposition["year"] == 2026,
            "annual_spend_per_customer"
        ].iloc[0],

        decomposition.loc[
            decomposition["year"] == 2026,
            "items_per_customer"
        ].iloc[0],

        decomposition.loc[
            decomposition["year"] == 2026,
            "price_per_item"
        ].iloc[0]
    ]
})

driver_yoy["yoy"] = (
    (
        driver_yoy["2026"]
        / driver_yoy["2025"]
        - 1
    ) * 100
).round(2)

display(driver_yoy)

In [ ]:
# =========================
# 23. Decomposition Validation
# =========================

sales_2025 = decomposition.loc[
    decomposition["year"] == 2025,
    "total_sales"
].iloc[0]

sales_2026 = decomposition.loc[
    decomposition["year"] == 2026,
    "total_sales"
].iloc[0]

customers_2025 = decomposition.loc[
    decomposition["year"] == 2025,
    "unique_customers"
].iloc[0]

customers_2026 = decomposition.loc[
    decomposition["year"] == 2026,
    "unique_customers"
].iloc[0]

spend_2025 = decomposition.loc[
    decomposition["year"] == 2025,
    "annual_spend_per_customer"
].iloc[0]

spend_2026 = decomposition.loc[
    decomposition["year"] == 2026,
    "annual_spend_per_customer"
].iloc[0]

# 売上 = 顧客数 × 1人あたり年間購入金額
assert np.isclose(
    sales_2025,
    customers_2025 * spend_2025
)

assert np.isclose(
    sales_2026,
    customers_2026 * spend_2026
)

print("✓ Revenue decomposition validation passed.")
print("Revenue = Customers × Annual Spend per Customer")

In [ ]:
# =========================
# 24. Customer Spend Decomposition
# =========================

decomposition["items_per_customer"] = (
    decomposition["total_quantity"]
    / decomposition["unique_customers"]
)

decomposition["price_per_item"] = (
    decomposition["total_sales"]
    / decomposition["total_quantity"]
)

# 検算
decomposition["calculated_spend_per_customer"] = (
    decomposition["items_per_customer"]
    * decomposition["price_per_item"]
)

display(
    decomposition[
        [
            "year",
            "annual_spend_per_customer",
            "items_per_customer",
            "price_per_item",
            "calculated_spend_per_customer"
        ]
    ].style.format({
        "annual_spend_per_customer": "${:,.2f}",
        "items_per_customer": "{:.2f}",
        "price_per_item": "${:,.2f}",
        "calculated_spend_per_customer": "${:,.2f}"
    })
)

In [ ]:
# =========================
# 25. Customer Spend Validation
# =========================

assert np.allclose(
    decomposition["annual_spend_per_customer"],
    decomposition["calculated_spend_per_customer"]
)

print(
    "✓ Customer spend decomposition validation passed."
)
print(
    "Annual Spend per Customer "
    "= Items per Customer × Price per Item"
)

In [ ]:
# =========================
# 26. Customer Spend Driver YoY
# =========================

spend_driver_yoy = pd.DataFrame({
    "metric": [
        "Annual Spend per Customer",
        "Items per Customer",
        "Price per Item"
    ],
    "2025": [
        decomposition.loc[
            decomposition["year"] == 2025,
            "annual_spend_per_customer"
        ].iloc[0],

        decomposition.loc[
            decomposition["year"] == 2025,
            "items_per_customer"
        ].iloc[0],

        decomposition.loc[
            decomposition["year"] == 2025,
            "price_per_item"
        ].iloc[0]
    ],
    "2026": [
        decomposition.loc[
            decomposition["year"] == 2026,
            "annual_spend_per_customer"
        ].iloc[0],

        decomposition.loc[
            decomposition["year"] == 2026,
            "items_per_customer"
        ].iloc[0],

        decomposition.loc[
            decomposition["year"] == 2026,
            "price_per_item"
        ].iloc[0]
    ]
})

spend_driver_yoy["yoy"] = (
    (
        spend_driver_yoy["2026"]
        / spend_driver_yoy["2025"]
        - 1
    ) * 100
).round(2)

display(spend_driver_yoy)

In [ ]:
# =========================
# 27. Customer Spend Driver Chart
# =========================

plot_data = spend_driver_yoy[
    spend_driver_yoy["metric"] != "Annual Spend per Customer"
].copy()

plt.figure(figsize=(9, 5))

plt.bar(
    plot_data["metric"],
    plot_data["yoy"]
)

plt.axhline(
    0,
    linewidth=1
)

plt.title(
    "YoY Change in Customer Spend Drivers"
)

plt.xlabel("Driver")
plt.ylabel("YoY Change (%)")

for i, value in enumerate(plot_data["yoy"]):
    va = "bottom" if value >= 0 else "top"

    plt.text(
        i,
        value,
        f"{value:+.1f}%",
        ha="center",
        va=va
    )

plt.tight_layout()
plt.show()

In [ ]:
# =========================
# 28. Fact / Hypothesis
# =========================

fact_hypothesis = pd.DataFrame({
    "category": [
        "Fact",
        "Fact",
        "Fact",
        "Fact",
        "Hypothesis",
        "Hypothesis"
    ],
    "finding": [
        "Total sales changed year over year.",
        "The number of unique customers changed year over year.",
        "Annual spend per customer changed year over year.",
        "Items per customer and price per item contributed to the change in annual spend per customer.",
        "A change in product mix may have affected items per customer.",
        "Promotion, assortment, or customer behavior may have affected purchasing frequency and basket size."
    ]
})

display(fact_hypothesis)

In [ ]:
# =========================
# 29. Extract Key Findings
# =========================

sales_yoy = yoy_2026["total_sales_yoy"].iloc[0]
customer_yoy = yoy_2026["unique_customers_yoy"].iloc[0]
spend_yoy = yoy_2026["annual_spend_per_customer_yoy"].iloc[0]

items_yoy = (
    spend_driver_yoy.loc[
        spend_driver_yoy["metric"] == "Items per Customer",
        "yoy"
    ].iloc[0]
)

price_yoy = (
    spend_driver_yoy.loc[
        spend_driver_yoy["metric"] == "Price per Item",
        "yoy"
    ].iloc[0]
)

print(f"Sales YoY: {sales_yoy:+.1f}%")
print(f"Unique Customers YoY: {customer_yoy:+.1f}%")
print(f"Annual Spend per Customer YoY: {spend_yoy:+.1f}%")
print(f"Items per Customer YoY: {items_yoy:+.1f}%")
print(f"Price per Item YoY: {price_yoy:+.1f}%")

In [ ]:
# =========================
# 30. Automated Fact Summary
# =========================

print("=== Fact Summary ===")

if sales_yoy > 0:
    print(f"• Sales increased by {sales_yoy:.1f}% YoY.")
else:
    print(f"• Sales decreased by {abs(sales_yoy):.1f}% YoY.")

if customer_yoy > 0:
    print(
        f"• Unique customers increased by "
        f"{customer_yoy:.1f}% YoY."
    )
else:
    print(
        f"• Unique customers decreased by "
        f"{abs(customer_yoy):.1f}% YoY."
    )

if spend_yoy > 0:
    print(
        f"• Annual spend per customer increased by "
        f"{spend_yoy:.1f}% YoY."
    )
else:
    print(
        f"• Annual spend per customer decreased by "
        f"{abs(spend_yoy):.1f}% YoY."
    )

print(
    f"• Items per customer changed by "
    f"{items_yoy:+.1f}% YoY."
)

print(
    f"• Price per item changed by "
    f"{price_yoy:+.1f}% YoY."
)

In [ ]:
# =========================
# 31. Insight Summary
# =========================

insights = pd.DataFrame({
    "insight_id": [
        "I001",
        "I002",
        "I003"
    ],
    "insight": [
        "Sales growth should be evaluated together with customer growth and customer spend.",
        "Changes in annual spend per customer can be further explained by items per customer and price per item.",
        "The current KPI analysis identifies where performance changed, but does not establish the underlying causal factors."
    ],
    "business_implication": [
        "Customer acquisition and customer monetization should be monitored separately.",
        "Increasing basket size or improving product value can be potential growth levers.",
        "Product mix, promotion, customer behavior, and purchasing frequency require additional analysis."
    ]
})

display(insights)

In [ ]:
# =========================
# 32. Fact / Insight / Hypothesis Framework
# =========================

analysis_framework = pd.DataFrame({
    "step": [
        "Fact",
        "Fact",
        "Fact",
        "Insight",
        "Insight",
        "Hypothesis"
    ],
    "content": [
        f"Sales YoY: {sales_yoy:+.1f}%",
        f"Unique Customers YoY: {customer_yoy:+.1f}%",
        f"Annual Spend per Customer YoY: {spend_yoy:+.1f}%",
        "Sales performance is driven by both customer count and spend per customer.",
        "Customer spend can be decomposed into items per customer and price per item.",
        "Product assortment, promotions, or purchasing behavior may explain changes in basket size."
    ]
})

display(analysis_framework)

In [ ]:
# =========================
# 33. Next Analysis Questions
# =========================

next_questions = pd.DataFrame({
    "priority": [
        1,
        2,
        3,
        4
    ],
    "analysis_question": [
        "Which product categories contribute most to sales growth?",
        "How does purchasing behavior differ between Japan and USA?",
        "Does the Online channel generate higher customer spend than Store?",
        "Which customer segments have the highest purchase value?"
    ]
})

display(next_questions)

In [ ]:
# =========================
# 34. Country × Channel KPI
# =========================

country_channel_kpi = (
    analysis_df
    .groupby(["country", "sales_channel"])
    .agg(
        total_sales=("sales_amount", "sum"),
        total_orders=("order_id", "nunique"),
        unique_customers=("customer_id", "nunique"),
        total_quantity=("quantity", "sum")
    )
    .reset_index()
)

country_channel_kpi["average_order_value"] = (
    country_channel_kpi["total_sales"]
    / country_channel_kpi["total_orders"]
)

country_channel_kpi["items_per_order"] = (
    country_channel_kpi["total_quantity"]
    / country_channel_kpi["total_orders"]
)

display(
    country_channel_kpi.round(2)
)

In [ ]:
# =========================
# 35. Country Sales Mix
# =========================

country_sales = (
    analysis_df
    .groupby("country")
    .agg(
        total_sales=("sales_amount", "sum"),
        total_orders=("order_id", "nunique"),
        unique_customers=("customer_id", "nunique")
    )
    .reset_index()
)

country_sales["sales_share"] = (
    country_sales["total_sales"]
    / country_sales["total_sales"].sum()
    * 100
)

country_sales["aov"] = (
    country_sales["total_sales"]
    / country_sales["total_orders"]
)

display(country_sales.round(2))

In [ ]:
# =========================
# 36. Country Sales Chart
# =========================

plt.figure(figsize=(8, 5))

plt.bar(
    country_sales["country"],
    country_sales["total_sales"]
)

plt.title("Total Sales by Country")
plt.xlabel("Country")
plt.ylabel("Sales (USD)")

for i, value in enumerate(country_sales["total_sales"]):
    plt.text(
        i,
        value,
        f"${value:,.0f}",
        ha="center",
        va="bottom"
    )

plt.tight_layout()
plt.show()

In [ ]:
# =========================
# 37. Channel Comparison
# =========================

channel_kpi = (
    analysis_df
    .groupby("sales_channel")
    .agg(
        total_sales=("sales_amount", "sum"),
        total_orders=("order_id", "nunique"),
        unique_customers=("customer_id", "nunique"),
        total_quantity=("quantity", "sum")
    )
    .reset_index()
)

channel_kpi["aov"] = (
    channel_kpi["total_sales"]
    / channel_kpi["total_orders"]
)

channel_kpi["items_per_order"] = (
    channel_kpi["total_quantity"]
    / channel_kpi["total_orders"]
)

display(channel_kpi.round(2))

In [ ]:
# =========================
# 38. Product Category KPI
# =========================

category_kpi = (
    analysis_df
    .groupby("category")
    .agg(
        total_sales=("sales_amount", "sum"),
        total_orders=("order_id", "nunique"),
        unique_customers=("customer_id", "nunique"),
        total_quantity=("quantity", "sum")
    )
    .reset_index()
)

category_kpi["sales_share"] = (
    category_kpi["total_sales"]
    / category_kpi["total_sales"].sum()
    * 100
)

category_kpi["aov"] = (
    category_kpi["total_sales"]
    / category_kpi["total_orders"]
)

category_kpi["items_per_order"] = (
    category_kpi["total_quantity"]
    / category_kpi["total_orders"]
)

category_kpi = category_kpi.sort_values(
    "total_sales",
    ascending=False
)

display(category_kpi.round(2))

In [ ]:
# =========================
# 39. Category Sales Ranking
# =========================

plt.figure(figsize=(10, 5))

plt.bar(
    category_kpi["category"],
    category_kpi["total_sales"]
)

plt.title("Sales by Product Category")
plt.xlabel("Product Category")
plt.ylabel("Sales (USD)")

plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()

In [ ]:
# =========================
# 40. Category Sales Mix
# =========================

category_mix = category_kpi[
    [
        "category",
        "total_sales",
        "sales_share"
    ]
].copy()

category_mix["sales_share"] = (
    category_mix["sales_share"].round(2)
)

display(category_mix)

In [ ]:
# =========================
# 41. Category Sales YoY
# =========================

category_year = (
    analysis_df
    .groupby(["category", "year"])
    .agg(
        total_sales=("sales_amount", "sum"),
        total_quantity=("quantity", "sum"),
        total_orders=("order_id", "nunique")
    )
    .reset_index()
)

category_sales_yoy = (
    category_year
    .pivot(
        index="category",
        columns="year",
        values="total_sales"
    )
    .reset_index()
)

category_sales_yoy.columns.name = None

category_sales_yoy["sales_yoy"] = (
    (
        category_sales_yoy[2026]
        / category_sales_yoy[2025]
        - 1
    ) * 100
)

category_sales_yoy = category_sales_yoy.sort_values(
    "sales_yoy",
    ascending=False
)

display(
    category_sales_yoy.round(2)
)

In [ ]:
# =========================
# 42. Category YoY Chart
# =========================

plt.figure(figsize=(10, 5))

plt.bar(
    category_sales_yoy["category"],
    category_sales_yoy["sales_yoy"]
)

plt.axhline(
    0,
    linewidth=1
)

plt.title("Category Sales YoY Change")
plt.xlabel("Product Category")
plt.ylabel("YoY Change (%)")

plt.xticks(rotation=45, ha="right")

for i, value in enumerate(category_sales_yoy["sales_yoy"]):
    va = "bottom" if value >= 0 else "top"

    plt.text(
        i,
        value,
        f"{value:+.1f}%",
        ha="center",
        va=va
    )

plt.tight_layout()
plt.show()

In [ ]:
# =========================
# 43. Category Sales Contribution
# =========================

category_sales_yoy["sales_change"] = (
    category_sales_yoy[2026]
    - category_sales_yoy[2025]
)

category_contribution = (
    category_sales_yoy[
        [
            "category",
            2025,
            2026,
            "sales_change",
            "sales_yoy"
        ]
    ]
    .sort_values(
        "sales_change",
        ascending=False
    )
)

display(
    category_contribution.round(2)
)

In [ ]:
# =========================
# 44. Category × Country KPI
# =========================

category_country_kpi = (
    analysis_df
    .groupby(["country", "category"])
    .agg(
        total_sales=("sales_amount", "sum"),
        total_orders=("order_id", "nunique"),
        total_quantity=("quantity", "sum"),
        unique_customers=("customer_id", "nunique")
    )
    .reset_index()
)

category_country_kpi["aov"] = (
    category_country_kpi["total_sales"]
    / category_country_kpi["total_orders"]
)

category_country_kpi["sales_share_within_country"] = (
    category_country_kpi["total_sales"]
    /
    category_country_kpi.groupby("country")["total_sales"].transform("sum")
    * 100
)

display(
    category_country_kpi.round(2)
)

In [ ]:
# =========================
# 45. Category Sales by Country
# =========================

category_country_pivot = (
    category_country_kpi
    .pivot(
        index="category",
        columns="country",
        values="total_sales"
    )
    .fillna(0)
)

display(
    category_country_pivot.round(2)
)

In [ ]:
# =========================
# 46. Category Mix by Country
# =========================

category_country_mix = (
    category_country_kpi[
        [
            "country",
            "category",
            "sales_share_within_country"
        ]
    ]
    .pivot(
        index="category",
        columns="country",
        values="sales_share_within_country"
    )
    .fillna(0)
)

display(
    category_country_mix.round(2)
)

In [ ]:
# =========================
# 47. Customer Purchase KPI
# =========================

customer_kpi = (
    analysis_df
    .groupby(
        [
            "customer_id",
            "country",
            "customer_age",
            "customer_gender"
        ]
    )
    .agg(
        total_sales=("sales_amount", "sum"),
        total_orders=("order_id", "nunique"),
        total_quantity=("quantity", "sum"),
        first_order_date=("order_date", "min"),
        last_order_date=("order_date", "max")
    )
    .reset_index()
)

customer_kpi["average_order_value"] = (
    customer_kpi["total_sales"]
    / customer_kpi["total_orders"]
)

customer_kpi["items_per_order"] = (
    customer_kpi["total_quantity"]
    / customer_kpi["total_orders"]
)

display(customer_kpi.head(10).round(2))

In [ ]:
# =========================
# 48. Customer Segmentation
# =========================

customer_kpi["customer_segment"] = pd.qcut(
    customer_kpi["total_sales"],
    q=3,
    labels=[
        "Low Value",
        "Mid Value",
        "High Value"
    ],
    duplicates="drop"
)

display(
    customer_kpi[
        [
            "customer_id",
            "country",
            "customer_age",
            "customer_gender",
            "total_sales",
            "total_orders",
            "customer_segment"
        ]
    ]
    .sort_values("total_sales", ascending=False)
    .head(20)
)

In [ ]:
# =========================
# 49. Segment KPI
# =========================

segment_kpi = (
    customer_kpi
    .groupby("customer_segment", observed=True)
    .agg(
        customers=("customer_id", "nunique"),
        total_sales=("total_sales", "sum"),
        total_orders=("total_orders", "sum"),
        average_customer_sales=("total_sales", "mean"),
        average_orders=("total_orders", "mean")
    )
    .reset_index()
)

segment_kpi["sales_share"] = (
    segment_kpi["total_sales"]
    / segment_kpi["total_sales"].sum()
    * 100
)

display(segment_kpi.round(2))

In [ ]:
# =========================
# 50. Segment Sales Chart
# =========================

plt.figure(figsize=(8, 5))

plt.bar(
    segment_kpi["customer_segment"].astype(str),
    segment_kpi["total_sales"]
)

plt.title("Sales by Customer Segment")
plt.xlabel("Customer Segment")
plt.ylabel("Sales (USD)")

plt.tight_layout()
plt.show()

In [ ]:
# =========================
# 51. Customer Age Groups
# =========================

customer_kpi["age_group"] = pd.cut(
    customer_kpi["customer_age"],
    bins=[0, 19, 29, 39, 49, 59, 69, 120],
    labels=[
        "Under 20",
        "20-29",
        "30-39",
        "40-49",
        "50-59",
        "60-69",
        "70+"
    ],
    right=True
)

display(
    customer_kpi[
        [
            "customer_id",
            "customer_age",
            "age_group",
            "total_sales",
            "total_orders"
        ]
    ].head(10)
)

In [ ]:
# =========================
# 52. Age Group KPI
# =========================

age_kpi = (
    customer_kpi
    .groupby("age_group", observed=True)
    .agg(
        customers=("customer_id", "nunique"),
        total_sales=("total_sales", "sum"),
        total_orders=("total_orders", "sum"),
        average_customer_sales=("total_sales", "mean"),
        average_orders=("total_orders", "mean")
    )
    .reset_index()
)

age_kpi["sales_share"] = (
    age_kpi["total_sales"]
    / age_kpi["total_sales"].sum()
    * 100
)

display(age_kpi.round(2))

In [ ]:
# =========================
# 53. Sales by Age Group
# =========================

plt.figure(figsize=(10, 5))

plt.bar(
    age_kpi["age_group"].astype(str),
    age_kpi["total_sales"]
)

plt.title("Sales by Customer Age Group")
plt.xlabel("Age Group")
plt.ylabel("Sales (USD)")

plt.xticks(rotation=30)

plt.tight_layout()
plt.show()

In [2]:
# =========================
# 54. Age Group × Country
# =========================

age_country_kpi = (
    customer_kpi
    .groupby(
        ["country", "age_group"],
        observed=True
    )
    .agg(
        customers=("customer_id", "nunique"),
        total_sales=("total_sales", "sum"),
        total_orders=("total_orders", "sum")
    )
    .reset_index()
)

age_country_kpi["sales_per_customer"] = (
    age_country_kpi["total_sales"]
    / age_country_kpi["customers"]
)

display(age_country_kpi.round(2))

NameError: name 'customer_kpi' is not defined

In [ ]:
# =========================
# 55. Executive Summary KPI
# =========================

executive_summary = pd.DataFrame({
    "KPI": [
        "Total Sales",
        "Total Orders",
        "Unique Customers",
        "Total Quantity",
        "Average Order Value",
        "Items per Order"
    ],
    "Value": [
        analysis_df["sales_amount"].sum(),
        analysis_df["order_id"].nunique(),
        analysis_df["customer_id"].nunique(),
        analysis_df["quantity"].sum(),
        analysis_df["sales_amount"].sum()
        / analysis_df["order_id"].nunique(),
        analysis_df["quantity"].sum()
        / analysis_df["order_id"].nunique()
    ]
})

display(executive_summary.round(2))

In [ ]:
# =========================
# 56. Year-over-Year KPI Summary
# =========================

year_kpi = (
    analysis_df
    .groupby("year")
    .agg(
        total_sales=("sales_amount", "sum"),
        total_orders=("order_id", "nunique"),
        unique_customers=("customer_id", "nunique"),
        total_quantity=("quantity", "sum")
    )
    .reset_index()
)

year_kpi["aov"] = (
    year_kpi["total_sales"]
    / year_kpi["total_orders"]
)

year_kpi["items_per_order"] = (
    year_kpi["total_quantity"]
    / year_kpi["total_orders"]
)

display(year_kpi.round(2))

In [ ]:
# =========================
# 57. Final YoY Summary
# =========================

yoy_summary = pd.DataFrame({
    "Metric": [
        "Sales",
        "Orders",
        "Unique Customers",
        "Quantity",
        "Average Order Value",
        "Items per Order"
    ],
    "YoY Change (%)": [
        (
            year_kpi.loc[year_kpi["year"] == 2026, "total_sales"].iloc[0]
            /
            year_kpi.loc[year_kpi["year"] == 2025, "total_sales"].iloc[0]
            - 1
        ) * 100,

        (
            year_kpi.loc[year_kpi["year"] == 2026, "total_orders"].iloc[0]
            /
            year_kpi.loc[year_kpi["year"] == 2025, "total_orders"].iloc[0]
            - 1
        ) * 100,

        (
            year_kpi.loc[year_kpi["year"] == 2026, "unique_customers"].iloc[0]
            /
            year_kpi.loc[year_kpi["year"] == 2025, "unique_customers"].iloc[0]
            - 1
        ) * 100,

        (
            year_kpi.loc[year_kpi["year"] == 2026, "total_quantity"].iloc[0]
            /
            year_kpi.loc[year_kpi["year"] == 2025, "total_quantity"].iloc[0]
            - 1
        ) * 100,

        (
            year_kpi.loc[year_kpi["year"] == 2026, "aov"].iloc[0]
            /
            year_kpi.loc[year_kpi["year"] == 2025, "aov"].iloc[0]
            - 1
        ) * 100,

        (
            year_kpi.loc[year_kpi["year"] == 2026, "items_per_order"].iloc[0]
            /
            year_kpi.loc[year_kpi["year"] == 2025, "items_per_order"].iloc[0]
            - 1
        ) * 100
    ]
})

display(yoy_summary.round(2))

In [ ]:
# =========================
# 58. Key Insights
# =========================

total_sales_2025 = year_kpi.loc[
    year_kpi["year"] == 2025, "total_sales"
].iloc[0]

total_sales_2026 = year_kpi.loc[
    year_kpi["year"] == 2026, "total_sales"
].iloc[0]

sales_yoy = (
    total_sales_2026
    / total_sales_2025
    - 1
) * 100

aov_2025 = year_kpi.loc[
    year_kpi["year"] == 2025, "aov"
].iloc[0]

aov_2026 = year_kpi.loc[
    year_kpi["year"] == 2026, "aov"
].iloc[0]

aov_yoy = (
    aov_2026
    / aov_2025
    - 1
) * 100

orders_2025 = year_kpi.loc[
    year_kpi["year"] == 2025, "total_orders"
].iloc[0]

orders_2026 = year_kpi.loc[
    year_kpi["year"] == 2026, "total_orders"
].iloc[0]

orders_yoy = (
    orders_2026
    / orders_2025
    - 1
) * 100

print("===== Key Insights =====")
print()
print(f"1. Sales YoY: {sales_yoy:+.2f}%")
print(f"2. Orders YoY: {orders_yoy:+.2f}%")
print(f"3. AOV YoY: {aov_yoy:+.2f}%")
print()

print("Top Categories by Sales:")
display(
    category_kpi[
        ["category", "total_sales", "sales_share"]
    ]
    .head(5)
    .round(2)
)

print("Country × Channel Performance:")
display(
    country_channel_kpi[
        [
            "country",
            "sales_channel",
            "total_sales",
            "total_orders",
            "aov"
        ]
    ]
    .sort_values("total_sales", ascending=False)
    .round(2)
)

In [ ]:
# =========================
# 59. Final Analysis Summary
# =========================

print("===================================")
print("       FINAL ANALYSIS SUMMARY")
print("===================================")
print()

print(f"Total Sales: ${analysis_df['sales_amount'].sum():,.2f}")
print(f"Total Orders: {analysis_df['order_id'].nunique():,}")
print(f"Unique Customers: {analysis_df['customer_id'].nunique():,}")
print(f"Total Quantity: {analysis_df['quantity'].sum():,}")
print(
    f"Average Order Value: "
    f"${analysis_df['sales_amount'].sum() / analysis_df['order_id'].nunique():,.2f}"
)

print()
print("Year-over-Year:")
print(f"Sales: {sales_yoy:+.2f}%")
print(f"Orders: {orders_yoy:+.2f}%")
print(f"AOV: {aov_yoy:+.2f}%")

print()
print("Top Selling Categories:")

for _, row in category_kpi.head(3).iterrows():
    print(
        f"- {row['category']}: "
        f"${row['total_sales']:,.2f}"
    )

In [ ]:
# =========================
# 60. Conclusion
# =========================

print("===================================")
print("             CONCLUSION")
print("===================================")
print()

print("1. Overall Performance")
print(
    f"Sales changed by {sales_yoy:+.2f}% from 2025 to 2026."
)
print(
    f"Orders changed by {orders_yoy:+.2f}%."
)
print(
    f"Average Order Value changed by {aov_yoy:+.2f}%."
)

print()
print("2. Product Performance")
top_category = category_kpi.iloc[0]

print(
    f"The highest-sales category was "
    f"{top_category['category']} "
    f"with ${top_category['total_sales']:,.2f} in sales."
)

print()
print("3. Customer Performance")
top_segment = segment_kpi.sort_values(
    "total_sales",
    ascending=False
).iloc[0]

print(
    f"The {top_segment['customer_segment']} segment "
    f"generated the largest share of sales."
)

print()
print("4. Market / Channel Performance")

top_country_channel = country_channel_kpi.sort_values(
    "total_sales",
    ascending=False
).iloc[0]

print(
    f"The strongest country-channel combination was "
    f"{top_country_channel['country']} / "
    f"{top_country_channel['sales_channel']}."
)

print()
print("5. Business Implication")
print(
    "The analysis suggests that sales performance should be "
    "evaluated not only at the total-sales level, but also by "
    "customer segment, product category, country, and sales channel."
)